# ML Training

## Luis Mateo Sanchez - 301339726

# Credit Default Model Training Demo

This notebook is a demonstration of the machine learning work completed in Amazon SageMaker Studio on AWS.

Its purpose is to show the main steps performed during the ML / Data Scientist phase of the project, including:

- loading the processed dataset from Amazon S3
- reviewing the dataset structure and target distribution
- splitting the data into training, validation, and test sets
- training and comparing multiple classification models
- evaluating model performance using metrics such as Accuracy, Precision, Recall, F1-score, and ROC-AUC
- selecting the final model based on performance on an imbalanced dataset
- saving the final model and configuration for deployment or handoff

This notebook is intended as a project demonstration and summary of the work executed in SageMaker Studio. It does not represent the full AWS production pipeline or infrastructure configuration.


## Project Summary

- **Dataset:** Processed credit default dataset stored in Amazon S3
- **Target Variable:** `default_payment`
- **Problem Type:** Binary classification
- **Goal:** Predict whether a customer will default on the next payment
- **Final Selected Model:** XGBoost
- **Model Selection Criteria:** F1-score, Recall, and ROC-AUC, due to class imbalance in the dataset

### Final Test Results
- **Accuracy:** 0.7649
- **Precision:** 0.4759
- **Recall:** 0.6135
- **F1-score:** 0.5360
- **ROC-AUC:** 0.7777


## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'crusq65de4ebs7'
os.environ['DataZoneDomainId'] = 'dzd-3lv269ot9spzdz'
os.environ['DataZoneEnvironmentId'] = '6aeyk48k85b4mf'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "crusq65de4ebs7",
                "DataZoneDomainId": "dzd-3lv269ot9spzdz",
                "DataZoneEnvironmentId": "6aeyk48k85b4mf",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


In [0]:
s3_path = "s3://cloud-machine-dataset-429293899944-us-east-1-an/processed_data/run-1776216373565-part-r-00000"

df = pd.read_csv(s3_path)
df.head()


,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_payment,SEX_male,SEX_female,EDU_grad_school,EDU_university,EDU_high_school,EDU_others,MARRIED,SINGLE,MARRIAGE_others
0,-1.136701,-1.245999,2,2,-1,-1,-2,-2,-0.642490,-0.647388,-0.667982,-0.672486,-0.663047,-0.652713,-0.341936,-0.227082,-0.296796,-0.308057,-0.314131,-0.293377,1,0,1,0,1,0,0,1,0,0
1,-0.365974,-1.029030,-1,2,0,0,0,2,-0.659208,-0.666735,-0.639244,-0.621626,-0.606219,-0.597956,-0.341936,-0.213584,-0.240001,-0.244226,-0.314131,-0.180875,1,0,1,0,1,0,0,0,1,0
2,-0.597192,-0.161154,0,0,0,0,0,0,-0.298555,-0.493891,-0.482400,-0.449723,-0.417181,-0.391623,-0.250287,-0.191884,-0.240001,-0.244226,-0.248679,-0.012122,0,0,1,0,1,0,0,0,1,0
3,-0.905483,0.164300,0,0,0,0,0,0,-0.057490,-0.013292,0.032846,-0.232369,-0.186726,-0.156576,-0.221187,-0.169358,-0.228641,-0.237842,-0.244162,-0.237126,0,0,1,0,1,0,0,1,0,0
4,-0.905483,2.333990,-1,0,-1,0,0,0,-0.578608,-0.611308,-0.161186,-0.346991,-0.348131,-0.331476,-0.221187,1.335012,0.271161,0.266429,-0.269034,-0.255183,0,1,0,0,1,0,0,1,0,0


In [0]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

Shape: (30000, 30)

Columns:
['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_payment', 'SEX_male', 'SEX_female', 'EDU_grad_school', 'EDU_university', 'EDU_high_school', 'EDU_others', 'MARRIED', 'SINGLE', 'MARRIAGE_others']

Data types:
LIMIT_BAL          float64
AGE                float64
PAY_0                int64
PAY_2                int64
PAY_3                int64
PAY_4                int64
PAY_5                int64
PAY_6                int64
BILL_AMT1          float64
BILL_AMT2          float64
BILL_AMT3          float64
BILL_AMT4          float64
BILL_AMT5          float64
BILL_AMT6          float64
PAY_AMT1           float64
PAY_AMT2           float64
PAY_AMT3           float64
PAY_AMT4           float64
PAY_AMT5           float64
PAY_AMT6           float64
default_payment      int64
SEX_male   

In [0]:
target_col = "default_payment"

print("Target distribution:")
print(df[target_col].value_counts())
print("\nTarget proportion:")
print(df[target_col].value_counts(normalize=True))

Target distribution:
default_payment
0    23364
1     6636
Name: count, dtype: int64

Target proportion:
default_payment
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [0]:
X = df.drop(columns=[target_col])
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (30000, 29)
y shape: (30000,)


In [0]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (21000, 29) (21000,)
Validation: (4500, 29) (4500,)
Test: (4500, 29) (4500,)


In [0]:
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)

y_val_pred = log_model.predict(X_val)
y_val_prob = log_model.predict_proba(X_val)[:, 1]

print("Logistic Regression - Validation Metrics")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_pred), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_prob), 4))

Logistic Regression - Validation Metrics
Accuracy : 0.8084
Precision: 0.6905
Recall   : 0.2422
F1 Score : 0.3586
ROC-AUC  : 0.7138


In [0]:
print(classification_report(y_val, y_val_pred))
print(confusion_matrix(y_val, y_val_pred))

              precision    recall  f1-score   support

           0       0.82      0.97      0.89      3505
           1       0.69      0.24      0.36       995

    accuracy                           0.81      4500
   macro avg       0.75      0.61      0.62      4500
weighted avg       0.79      0.81      0.77      4500

[[3397  108]
 [ 754  241]]


In [0]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_val_pred_rf = rf_model.predict(X_val)
y_val_prob_rf = rf_model.predict_proba(X_val)[:, 1]

print("Random Forest - Validation Metrics")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred_rf), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_rf), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred_rf), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred_rf), 4))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_prob_rf), 4))

Random Forest - Validation Metrics
Accuracy : 0.814
Precision: 0.6411
Recall   : 0.3608
F1 Score : 0.4617
ROC-AUC  : 0.7572


In [0]:
print(classification_report(y_val, y_val_pred_rf))
print(confusion_matrix(y_val, y_val_pred_rf))

              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3505
           1       0.64      0.36      0.46       995

    accuracy                           0.81      4500
   macro avg       0.74      0.65      0.67      4500
weighted avg       0.79      0.81      0.79      4500

[[3304  201]
 [ 636  359]]


### Model Selected: RandomForestClassifier

In [0]:
best_model = rf_model

y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:, 1]

print("Test Metrics")
print("Accuracy :", round(accuracy_score(y_test, y_test_pred), 4))
print("Precision:", round(precision_score(y_test, y_test_pred), 4))
print("Recall   :", round(recall_score(y_test, y_test_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_test_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob), 4))

Test Metrics
Accuracy : 0.8162
Precision: 0.6444
Recall   : 0.3785
F1 Score : 0.4769
ROC-AUC  : 0.7643


In [0]:
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3504
           1       0.64      0.38      0.48       996

    accuracy                           0.82      4500
   macro avg       0.74      0.66      0.68      4500
weighted avg       0.80      0.82      0.80      4500

[[3296  208]
 [ 619  377]]


In [0]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance.head(15)

,feature,importance
2,PAY_0,0.098847
1,AGE,0.064419
0,LIMIT_BAL,0.059589
8,BILL_AMT1,0.057398
9,BILL_AMT2,0.052694
14,PAY_AMT1,0.050340
13,BILL_AMT6,0.049615
10,BILL_AMT3,0.049167
12,BILL_AMT5,0.048752
11,BILL_AMT4,0.048273


In [0]:
import joblib

joblib.dump(best_model, "best_model.joblib")
print("Model saved as best_model.joblib")

Model saved as best_model.joblib


RandomForest with class_weight="balanced" and different thresholds 

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


In [0]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_balanced.fit(X_train, y_train)

y_val_prob_bal = rf_balanced.predict_proba(X_val)[:, 1]
y_val_pred_bal = (y_val_prob_bal >= 0.5).astype(int)

print("Balanced Random Forest - Validation Metrics")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred_bal), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_bal), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred_bal), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred_bal), 4))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_prob_bal), 4))


Balanced Random Forest - Validation Metrics
Accuracy : 0.8031
Precision: 0.563
Recall   : 0.4894
F1 Score : 0.5237
ROC-AUC  : 0.7702


In [0]:
print(classification_report(y_val, y_val_pred_bal))
print(confusion_matrix(y_val, y_val_pred_bal))


              precision    recall  f1-score   support

           0       0.86      0.89      0.88      3505
           1       0.56      0.49      0.52       995

    accuracy                           0.80      4500
   macro avg       0.71      0.69      0.70      4500
weighted avg       0.79      0.80      0.80      4500

[[3127  378]
 [ 508  487]]


In [0]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

results = []
for t in thresholds:
    preds = (y_val_prob_bal >= t).astype(int)
    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_val, preds),
        "precision": precision_score(y_val, preds),
        "recall": recall_score(y_val, preds),
        "f1": f1_score(y_val, preds)
    })

pd.DataFrame(results).sort_values(by="f1", ascending=False)


,threshold,accuracy,precision,recall,f1
3,0.45,0.795111,0.536757,0.535678,0.536217
2,0.40,0.778889,0.500000,0.573869,0.534394
1,0.35,0.750889,0.453744,0.621106,0.524395
4,0.50,0.803111,0.563006,0.489447,0.523656
5,0.55,0.811778,0.600271,0.445226,0.511252
0,0.30,0.702889,0.398577,0.675377,0.501305


Best Threshold: 0.55


In [0]:
best_threshold = 0.55

y_test_prob_bal = rf_balanced.predict_proba(X_test)[:, 1]
y_test_pred_bal = (y_test_prob_bal >= best_threshold).astype(int)

print("Balanced Random Forest - Test Metrics")
print("Accuracy :", round(accuracy_score(y_test, y_test_pred_bal), 4))
print("Precision:", round(precision_score(y_test, y_test_pred_bal), 4))
print("Recall   :", round(recall_score(y_test, y_test_pred_bal), 4))
print("F1 Score :", round(f1_score(y_test, y_test_pred_bal), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob_bal), 4))


Balanced Random Forest - Test Metrics
Accuracy : 0.8087
Precision: 0.5878
Recall   : 0.4538
F1 Score : 0.5122
ROC-AUC  : 0.7783


XGBoost

In [0]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [0]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", round(scale_pos_weight, 4))


scale_pos_weight: 3.521


In [0]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [0]:
y_val_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]
y_val_pred_xgb = (y_val_prob_xgb >= 0.5).astype(int)

print("XGBoost - Validation Metrics")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred_xgb), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_xgb), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred_xgb), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred_xgb), 4))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_prob_xgb), 4))

XGBoost - Validation Metrics
Accuracy : 0.7644
Precision: 0.474
Recall   : 0.596
F1 Score : 0.528
ROC-AUC  : 0.7736


In [0]:
print(classification_report(y_val, y_val_pred_xgb))
print(confusion_matrix(y_val, y_val_pred_xgb))


              precision    recall  f1-score   support

           0       0.88      0.81      0.84      3505
           1       0.47      0.60      0.53       995

    accuracy                           0.76      4500
   macro avg       0.68      0.70      0.69      4500
weighted avg       0.79      0.76      0.77      4500

[[2847  658]
 [ 402  593]]


In [0]:
thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

xgb_results = []
for t in thresholds:
    preds = (y_val_prob_xgb >= t).astype(int)
    xgb_results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_val, preds),
        "precision": precision_score(y_val, preds),
        "recall": recall_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, y_val_prob_xgb)
    })

pd.DataFrame(xgb_results).sort_values(by="f1", ascending=False)

,threshold,accuracy,precision,recall,f1,roc_auc
5,0.50,0.764444,0.474021,0.595980,0.528050,0.773581
6,0.55,0.783333,0.509416,0.543719,0.526009,0.773581
4,0.45,0.735333,0.433333,0.640201,0.516836,0.773581
3,0.40,0.692222,0.390572,0.699497,0.501260,0.773581
2,0.35,0.636444,0.355043,0.788945,0.489707,0.773581
1,0.30,0.570444,0.321265,0.847236,0.465875,0.773581
0,0.25,0.486889,0.287379,0.892462,0.434761,0.773581


Threshold: 0.50

In [0]:
best_threshold_xgb = 0.50

y_test_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_test_pred_xgb = (y_test_prob_xgb >= best_threshold_xgb).astype(int)

print("XGBoost - Test Metrics")
print("Accuracy :", round(accuracy_score(y_test, y_test_pred_xgb), 4))
print("Precision:", round(precision_score(y_test, y_test_pred_xgb), 4))
print("Recall   :", round(recall_score(y_test, y_test_pred_xgb), 4))
print("F1 Score :", round(f1_score(y_test, y_test_pred_xgb), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob_xgb), 4))


XGBoost - Test Metrics
Accuracy : 0.7649
Precision: 0.4759
Recall   : 0.6135
F1 Score : 0.536
ROC-AUC  : 0.7777


In [0]:
print(classification_report(y_test, y_test_pred_xgb))
print(confusion_matrix(y_test, y_test_pred_xgb))


              precision    recall  f1-score   support

           0       0.88      0.81      0.84      3504
           1       0.48      0.61      0.54       996

    accuracy                           0.76      4500
   macro avg       0.68      0.71      0.69      4500
weighted avg       0.79      0.76      0.77      4500

[[2831  673]
 [ 385  611]]


In [0]:
xgb_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(by="importance", ascending=False)

xgb_importance.head(15)


,feature,importance
2,PAY_0,0.307772
3,PAY_2,0.115534
5,PAY_4,0.052636
4,PAY_3,0.046095
7,PAY_6,0.033590
6,PAY_5,0.033107
15,PAY_AMT2,0.026079
0,LIMIT_BAL,0.023942
8,BILL_AMT1,0.022475
14,PAY_AMT1,0.021188


Since the dataset is imbalanced, model selection was based primarily on F1-score, recall, and ROC-AUC rather than accuracy alone. The final selected model was XGBoost, which achieved 0.7649 accuracy, 0.4759 precision, 0.6135 recall, 0.5360 F1-score, and 0.7777 ROC-AUC on the test set. Although its accuracy was lower than Random Forest, it identified substantially more default cases and provided the best balance for the positive class.

In [0]:
import joblib
import json

final_model = xgb_model
final_threshold = 0.50

joblib.dump(final_model, "xgboost_final_model.joblib")

model_config = {
    "model_name": "XGBoost Credit Default Classifier",
    "target_column": "default_payment",
    "threshold": final_threshold,
    "test_metrics": {
        "accuracy": 0.7649,
        "precision": 0.4759,
        "recall": 0.6135,
        "f1_score": 0.5360,
        "roc_auc": 0.7777
    },
    "feature_count": X_train.shape[1],
    "features": X_train.columns.tolist()
}

with open("xgboost_model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print("Saved:")
print("- xgboost_final_model.joblib")
print("- xgboost_model_config.json")


Saved:
- xgboost_final_model.joblib
- xgboost_model_config.json


Upload to S3

In [0]:
import boto3

bucket_name = "cloud-machine-dataset-429293899944-us-east-1-an"
model_key = "ml_models/xgboost_final_model.joblib"
config_key = "ml_models/xgboost_model_config.json"

s3 = boto3.client("s3")

s3.upload_file("xgboost_final_model.joblib", bucket_name, model_key)
s3.upload_file("xgboost_model_config.json", bucket_name, config_key)

print("Uploaded to S3:")
print(f"s3://{bucket_name}/{model_key}")
print(f"s3://{bucket_name}/{config_key}")


Uploaded to S3:
s3://cloud-machine-dataset-429293899944-us-east-1-an/ml_models/xgboost_final_model.joblib
s3://cloud-machine-dataset-429293899944-us-east-1-an/ml_models/xgboost_model_config.json


In [0]:
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="ml_models/")
for obj in response.get("Contents", []):
    print(obj["Key"])


ml_models/xgboost_final_model.joblib
ml_models/xgboost_model_config.json


In [0]:
model_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.8084,
        "Precision": 0.6905,
        "Recall": 0.2422,
        "F1 Score": 0.3586,
        "ROC-AUC": 0.7138
    },
    {
        "Model": "Random Forest",
        "Accuracy": 0.8162,
        "Precision": 0.6444,
        "Recall": 0.3785,
        "F1 Score": 0.4769,
        "ROC-AUC": 0.7643
    },
    {
        "Model": "Balanced Random Forest",
        "Accuracy": 0.8087,
        "Precision": 0.5878,
        "Recall": 0.4538,
        "F1 Score": 0.5122,
        "ROC-AUC": 0.7783
    },
    {
        "Model": "XGBoost",
        "Accuracy": 0.7649,
        "Precision": 0.4759,
        "Recall": 0.6135,
        "F1 Score": 0.5360,
        "ROC-AUC": 0.7777
    }
])

model_comparison.sort_values(by="F1 Score", ascending=False)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
3,XGBoost,0.7649,0.4759,0.6135,0.5360,0.7777
2,Balanced Random Forest,0.8087,0.5878,0.4538,0.5122,0.7783
1,Random Forest,0.8162,0.6444,0.3785,0.4769,0.7643
0,Logistic Regression,0.8084,0.6905,0.2422,0.3586,0.7138


In [0]:
model_comparison.sort_values(by="F1 Score", ascending=False).style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1 Score": "{:.4f}",
    "ROC-AUC": "{:.4f}"
})


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
3,XGBoost,0.7649,0.4759,0.6135,0.5360,0.7777
2,Balanced Random Forest,0.8087,0.5878,0.4538,0.5122,0.7783
1,Random Forest,0.8162,0.6444,0.3785,0.4769,0.7643
0,Logistic Regression,0.8084,0.6905,0.2422,0.3586,0.7138


XGBoost was selected as the final model because it achieved the highest F1-score and recall on the test set, which are especially important for this imbalanced credit default classification problem.

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()